# Week 5 RAG Optimisation - Local Preparation Run

**Status:** framework and local CPU smoke test, not the final Week 5 ablation.

This notebook validates the registered 18-configuration factorial design and exercises it with locally cached small models. It does not use RunPod or an external API. The production BGE-M3 / BGE cross-encoder / Llama stack, the frozen Senpai subset, and validated Faithfulness, Relevance, and Coverage evaluation remain deferred. Local metrics are deterministic lexical proxies and the MiniLM second stage is a bi-encoder smoke surrogate, not a cross-encoder result.

In [1]:
import os
import warnings
from pathlib import Path

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore', message='IProgress not found.*')

import pandas as pd
from W05_RAG_Optimisation import (
    LocalSmallModels,
    build_factorial_matrix,
    load_yaml,
    run_local_smoke,
    validate_config,
)

HERE = Path.cwd()
CONFIG_PATH = HERE / 'W05_RAG_Optimisation_Config_v0.1.0.yaml'
FIXTURE_PATH = HERE / 'W05_RAG_Local_Smoke_Fixture_v0.1.0.yaml'
config = load_yaml(CONFIG_PATH)
fixture = load_yaml(FIXTURE_PATH)
audit = validate_config(config)
pd.DataFrame([
    {
        'variant_id': variant.variant_id,
        'chunk_size_tokens': variant.chunk_size_tokens,
        'top_k': variant.top_k,
        'requested_reranking': variant.reranking,
    }
    for variant in build_factorial_matrix(config)
])

,variant_id,chunk_size_tokens,top_k,requested_reranking
0,chunk-256_topk-1_rerank-ce,256,1,cross_encoder
1,chunk-256_topk-1_rerank-none,256,1,none
2,chunk-256_topk-3_rerank-ce,256,3,cross_encoder
3,chunk-256_topk-3_rerank-none,256,3,none
4,chunk-256_topk-5_rerank-ce,256,5,cross_encoder
5,chunk-256_topk-5_rerank-none,256,5,none
6,chunk-512_topk-1_rerank-ce,512,1,cross_encoder
7,chunk-512_topk-1_rerank-none,512,1,none
8,chunk-512_topk-3_rerank-ce,512,3,cross_encoder
9,chunk-512_topk-3_rerank-none,512,3,none


## Local offline integration test

The runner loads the exact cached revisions recorded in the config: `google/flan-t5-small` for generation and `sentence-transformers/all-MiniLM-L6-v2` for a clearly labeled reranking surrogate. Model load and one warm-up request are excluded from request latency; the 18 variants run in a seed-42 randomized order.

In [2]:
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

disable_progress_bars()
transformers_logging.set_verbosity_error()
models = LocalSmallModels(config)
payload = run_local_smoke(config, fixture, models)
{
    'status': payload['status'],
    'generator': f"{payload['generator']['model_id']}@{payload['generator']['model_revision']}",
    'evaluation_set': payload['evaluation_set'],
    'rows': payload['row_count'],
    'variants': len(payload['variant_summaries']),
    'matched_contrasts': payload['design_audit']['matched_contrast_count'],
    'chunk_counts': payload['chunk_counts'],
    'metric_status': config['local_smoke']['metric_status'],
}

{'status': 'local_cpu_smoke_complete_not_final_week5_result',
 'generator': 'google/flan-t5-small@0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab',
 'evaluation_set': {'id': 'w05_senpai_synthetic_smoke',
  'version': '0.1.0',
  'items': 3},
 'rows': 54,
 'variants': 18,
 'matched_contrasts': 45,
 'chunk_counts': {'256': 15, '512': 9, '1024': 6},
 'metric_status': 'deterministic_lexical_proxies_not_ragas'}

In [3]:
summary = pd.DataFrame(payload['variant_summaries'])
summary[[
    'variant_id',
    'mean_lexical_context_support_proxy',
    'mean_reference_token_f1_proxy',
    'mean_required_term_coverage_proxy',
    'mean_question_to_response_ms',
]].sort_values('variant_id')

,variant_id,mean_lexical_context_support_proxy,mean_reference_token_f1_proxy,mean_required_term_coverage_proxy,mean_question_to_response_ms
0,chunk-1024_topk-1_rerank-ce,0.972222,0.024691,0.000000,615.375
1,chunk-1024_topk-1_rerank-none,0.972222,0.024691,0.000000,394.193
2,chunk-1024_topk-3_rerank-ce,1.000000,0.031746,0.066667,662.165
3,chunk-1024_topk-3_rerank-none,0.714286,0.060732,0.150000,299.149
4,chunk-1024_topk-5_rerank-ce,0.952381,0.000000,0.000000,979.402
5,chunk-1024_topk-5_rerank-none,0.866667,0.000000,0.000000,689.170
6,chunk-256_topk-1_rerank-ce,0.916667,0.039216,0.083333,575.593
7,chunk-256_topk-1_rerank-none,0.666667,0.000000,0.000000,293.208
8,chunk-256_topk-3_rerank-ce,0.972222,0.024691,0.000000,800.436
9,chunk-256_topk-3_rerank-none,0.888889,0.031746,0.066667,280.471


In [4]:
metric_ranges = {
    'reference_f1_proxy_min': float(round(summary['mean_reference_token_f1_proxy'].min(), 6)),
    'reference_f1_proxy_max': float(round(summary['mean_reference_token_f1_proxy'].max(), 6)),
    'required_coverage_proxy_min': float(round(summary['mean_required_term_coverage_proxy'].min(), 6)),
    'required_coverage_proxy_max': float(round(summary['mean_required_term_coverage_proxy'].max(), 6)),
}
metric_ranges

{'reference_f1_proxy_min': 0.0,
 'reference_f1_proxy_max': 0.15003,
 'required_coverage_proxy_min': 0.0,
 'required_coverage_proxy_max': 0.15}

## Interpretation

The smoke run is successful if all 54 rows satisfy the traceability contract, all 18 variants are present, and chunk counts differ across the three registered sizes. Low proxy answer quality is expected from this deliberately small local generator and makes the smoke-only Pareto frontier unsuitable for configuration selection. A final claim requires the frozen Senpai data, the true cross-encoder, calibrated evaluation coverage, row-level review, and comparable warm-path timing.

## Production continuation gates

1. Accept the exact Week 3 Senpai source/evaluation versions and preserve their metadata gates.
2. Execute the registered 18 variants with the pinned BGE-M3 and BGE reranker, holding the generator, prompt, evaluator, seed, and hardware constant.
3. Compute Faithfulness, Answer Relevance, and Required-point Coverage with finite-row coverage and calibration status.
4. Estimate factor effects only from matched pairs that differ in one variable; use the full grid for Pareto selection.
5. Preserve row-level inputs, retrieved chunks, outputs, revisions, seed, latency stages, invalid reasons, and human-review strata before writing the final Week 5 finding.